# EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria
- **Accuracy**: Correct information and calculations
- **Relevance**: Responses address the user's question
- **Completeness**: Comprehensive answers with actionable advice
- **Tool Usage**: Appropriate use of available tools
- **Reasoning**: Clear explanation of recommendations


## 1. Import and Initialize

In [1]:
from datetime import datetime
from agent import Agent

In [2]:
## TODO: Create the agent's instructions


ECOHOME_SYSTEM_PROMPT = """
You are the EcoHome Energy Advisor, an AI-powered assistant that helps households
optimize electricity usage to reduce cost and environmental impact.

Your responsibilities:
- Provide personalized, practical energy recommendations for homes with
  solar panels, electric vehicles, smart thermostats, and appliances.
- Optimize for BOTH electricity cost savings and carbon footprint reduction.
- Base your advice on available data such as:
  - Weather forecasts and solar irradiance
  - Time-of-use electricity pricing
  - Historical energy usage and solar generation
  - Proven energy-saving best practices from retrieved documents

How you should operate:
- Use tools whenever relevant to retrieve weather, pricing, usage history,
  solar generation data, or energy-saving tips.
- Combine insights from multiple tools before making recommendations.
- When suggesting optimizations (e.g., changing run time, charging schedule),
  explain the reasoning clearly and quantify potential savings when possible.
- Prefer energy usage during high solar generation and low-price periods.
- Be concise, clear, and actionable. Avoid generic advice.

Response style:
- Start with a direct answer to the user’s question.
- Explain WHY the recommendation is optimal (pricing, solar, history, or efficiency).
- Include estimated cost or energy savings if applicable.
- Use a helpful, professional, and friendly tone.
- If data is limited or assumptions are made, state them clearly.

Constraints:
- Do not invent data. Rely on tool outputs and reasonable assumptions.
- Do not mention internal implementation details, code, or tool names explicitly.
- If a question cannot be answered with available data, explain what information
  is missing and provide best-effort guidance.

Your goal is to help users save money, use cleaner energy, and make smarter
day-to-day energy decisions.
"""



In [3]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)

In [4]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context="Location: San Francisco, CA"
)

In [6]:
print(response["messages"][-1].content)

To minimize costs and maximize solar power for charging your electric vehicle (EV) tomorrow (October 7, 2023), I recommend charging your EV during the following time slots:

### Optimal Charging Times:
- **From 10 AM to 2 PM**: This period has the highest solar irradiance, which means your solar panels will generate the most electricity. 
- **From 10 AM to 12 PM**: The electricity rates during this time are relatively low, ranging from $0.116 to $0.14 per kWh.
- **From 12 PM to 2 PM**: The rates are slightly higher, around $0.114 to $0.129 per kWh, but the solar generation is still significant.

### Why This is Optimal:
1. **Solar Generation**: The solar irradiance is highest between 10 AM and 2 PM, allowing you to utilize the clean energy generated by your solar panels.
2. **Electricity Pricing**: The rates during these hours are lower compared to the peak hours (4 PM to 7 PM), where rates can go up to $0.184 per kWh.

### Estimated Savings:
- Charging during the recommended hours can

In [7]:
print("TOOLS:")
for msg in response["messages"]:
    obj = msg.model_dump()
    if obj.get("tool_call_id"):
        print("-", msg.name)

TOOLS:
- get_weather_forecast
- get_electricity_prices


## 2. Define Test Cases

In [11]:
# TODO: Define comprehensive test cases for the Energy Advisor
# Create 10 test cases covering different scenarios:
# - EV charging optimization
# - Thermostat settings
# - Appliance scheduling
# - Solar power maximization
# - Cost savings calculations

In [8]:

test_cases = [
    {
        "id": "ev_charging_1",
        "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "The response should recommend optimal charging time considering solar availability and electricity prices."
    },
    {
        "id": "ev_charging_2",
        "question": "Should I charge my EV during the day or at night if I want the lowest electricity cost?",
        "expected_tools": ["get_electricity_prices"],
        "expected_response": "The response should compare day vs night rates and suggest the cheaper option."
    },
    {
        "id": "thermostat_1",
        "question": "What thermostat settings should I use tomorrow to reduce cooling costs?",
        "expected_tools": ["get_weather_forecast"],
        "expected_response": "The response should suggest temperature settings based on weather conditions to reduce HVAC usage."
    },
    {
        "id": "thermostat_2",
        "question": "My AC is running a lot. How can I reduce HVAC energy consumption?",
        "expected_tools": ["query_energy_usage", "search_energy_tips"],
        "expected_response": "The response should identify HVAC usage patterns and recommend energy-saving actions."
    },
    {
        "id": "appliance_1",
        "question": "When is the best time today to run my dishwasher to save money?",
        "expected_tools": ["get_electricity_prices"],
        "expected_response": "The response should recommend running the appliance during off-peak or low-cost hours."
    },
    {
        "id": "appliance_2",
        "question": "Should I run my washing machine now or later to save electricity cost?",
        "expected_tools": ["get_electricity_prices"],
        "expected_response": "The response should compare current and later electricity rates and recommend the optimal time."
    },
    {
        "id": "solar_1",
        "question": "How can I maximize the use of my solar power today?",
        "expected_tools": ["get_weather_forecast", "query_solar_generation"],
        "expected_response": "The response should suggest shifting usage to high solar generation hours."
    },
    {
        "id": "solar_2",
        "question": "Is today a good day to rely more on solar energy for household usage?",
        "expected_tools": ["get_weather_forecast"],
        "expected_response": "The response should assess solar potential based on weather conditions."
    },
    {
        "id": "cost_savings_1",
        "question": "How much money can I save by shifting EV charging from peak to off-peak hours?",
        "expected_tools": ["get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "The response should estimate cost savings and explain the calculation assumptions."
    },
    {
        "id": "summary_1",
        "question": "Can you summarize my energy usage over the last 24 hours?",
        "expected_tools": ["get_recent_energy_summary"],
        "expected_response": "The response should provide a concise summary of consumption, cost, and major contributing devices."
    }
]


if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")

## 3. Run Agent Tests

In [9]:
CONTEXT = "Location: San Francisco, CA"

In [10]:
# Run the agent tests
# For each test case, call the agent and collect the response
# Store results for evaluation

print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)
    
    try:
        # Call the agent
        response = ecohome_agent.invoke(
            question=test_case['question'],
            context=CONTEXT
        )
        
        # Store the result
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': response,
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat()
        }
        test_results.append(result)
                
    except Exception as e:
        print(f"Error: {e}")
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': f"Error: {str(e)}",
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat(),
            'error': str(e)
        }
        test_results.append(result)

print(f"\nCompleted {len(test_results)} tests")


=== Running Agent Tests ===

Test 1: ev_charging_1
Question: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
--------------------------------------------------

Test 2: ev_charging_2
Question: Should I charge my EV during the day or at night if I want the lowest electricity cost?
--------------------------------------------------

Test 3: thermostat_1
Question: What thermostat settings should I use tomorrow to reduce cooling costs?
--------------------------------------------------

Test 4: thermostat_2
Question: My AC is running a lot. How can I reduce HVAC energy consumption?
--------------------------------------------------

Test 5: appliance_1
Question: When is the best time today to run my dishwasher to save money?
--------------------------------------------------

Test 6: appliance_2
Question: Should I run my washing machine now or later to save electricity cost?
--------------------------------------------------

Test 7: solar_1
Questio

In [11]:
test_results

[{'test_id': 'ev_charging_1',
  'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?',
  'response': {'messages': [SystemMessage(content='Location: San Francisco, CA', additional_kwargs={}, response_metadata={}, id='5df34afb-b1f1-4b74-9ee5-28891dbdfebc'),
    HumanMessage(content='When should I charge my electric car tomorrow to minimize cost and maximize solar power?', additional_kwargs={}, response_metadata={}, id='de1fa73b-31ef-401d-90e6-7e36f8e9f4c1'),
    AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_tvqc9dN5GsNJJbg7PnaOYkTA', 'function': {'arguments': '{"location": "San Francisco, CA", "days": 1}', 'name': 'get_weather_forecast'}, 'type': 'function'}, {'id': 'call_xp9BK5uqbY8N9n7e44p82jtt', 'function': {'arguments': '{"date": "2023-10-07"}', 'name': 'get_electricity_prices'}, 'type': 'function'}, {'id': 'call_yQVhSz4JMy9PIYrY8U6KzCf1', 'function': {'arguments': '{"start_date": "2023-10-07", "end_date": "2023-

## 4. Evaluate Responses

In [ ]:
# TODO: Implement evaluation functions
# Create functions to evaluate:
# - Final Response
# - Tool usage

In [12]:
# Helper functions

import re
import json
from typing import Any, Dict, List, Tuple

STOPWORDS = {
    "the","and","that","this","with","from","your","you","should","would","could",
    "what","when","how","can","is","are","to","of","in","for","on","a","an","it","be",
    "as","at","or","if","i","my","me"
}

def tokenize(text: str) -> List[str]:
    tokens = re.findall(r"[a-zA-Z0-9\.\$%]+", (text or "").lower())
    return [t for t in tokens if t not in STOPWORDS and len(t) > 2]

def extract_final_answer_text(response_obj: Any) -> str:
    """
    Your test_results[i]['response'] is a dict with response_obj['messages'].
    This extracts the LAST non-empty message.content (usually final AIMessage).
    """
    if isinstance(response_obj, str):
        return response_obj.strip()
    if not isinstance(response_obj, dict):
        return ""
    msgs = response_obj.get("messages", [])
    for msg in reversed(msgs):
        content = getattr(msg, "content", None)
        if isinstance(content, str) and content.strip():
            return content.strip()
    return ""

def extract_used_tools_from_messages(messages_list: List[Any]) -> List[str]:
    """
    Matches the notebook pattern: identify ToolMessage by tool_call_id and use msg.name. [1](https://github.com/MariusJochheim/Udacity-Master-Energy-Advisor/blob/main/03_run_and_evaluate.ipynb)
    """
    used = []
    for msg in messages_list or []:
        tool_name = getattr(msg, "name", None)
        tool_call_id = None
        try:
            obj = msg.model_dump()
            tool_call_id = obj.get("tool_call_id")
        except Exception:
            tool_call_id = getattr(msg, "tool_call_id", None)
        if tool_call_id and tool_name:
            used.append(tool_name)
    return used

def parse_expected_aspects(expected_response: str) -> Dict[str, bool]:
    """
    Reads the expected_response description and determines which aspects should appear.
    Example: 'time recommendation, cost analysis and solar consideration'
    """
    t = (expected_response or "").lower()
    return {
        "time": any(k in t for k in ["time", "schedule", "window", "hour"]),
        "cost": any(k in t for k in ["cost", "price", "rate", "savings", "usd", "$"]),
        "solar": any(k in t for k in ["solar", "irradiance", "sun", "generation"]),
        "actions": any(k in t for k in ["action", "steps", "recommend", "suggest"])
    }

def detect_aspects_in_response(response_text: str) -> Dict[str, bool]:
    t = (response_text or "").lower()
    return {
        "time": bool(re.search(r"\b(off-peak|peak|mid-peak|am|pm|hour|hours|time|window|schedule)\b", t)),
        "cost": bool(re.search(r"(\$[0-9]+(\.[0-9]+)?)|(\brate\b)|(\bper kwh\b)|(\bsave\b)|(\bsavings\b)", t)),
        "solar": bool(re.search(r"\b(solar|irradiance|sunny|generation|pv)\b", t)),
        "actions": bool(re.search(r"\b(recommend|set|avoid|schedule|run|charge|shift|try|steps|do this)\b", t)),
        "quantified": bool(re.search(r"(\$[0-9])|([0-9]+(\.[0-9]+)?\s*(kwh|%|hours?))", t))
    }

def clamp(x: float, lo: float = 0.0, hi: float = 100.0) -> float:
    return max(lo, min(hi, x))


In [19]:

def extract_final_response(response_obj):
    """
    Extract the final natural-language response from an agent response object.
    """
    if not isinstance(response_obj, dict):
        return None

    messages = response_obj.get("messages", [])
    if not messages:
        return None

    # Traverse messages from the end to find the final AIMessage with content
    for msg in reversed(messages):
        if hasattr(msg, "content") and isinstance(msg.content, str) and msg.content.strip():
            return msg.content

    return None


In [13]:
# TODO: Create a response evaluator


def evaluate_response(question: str, response_obj: Any, expected_response: str) -> Dict[str, Any]:
    """
    Required metrics (0–100):
      - ACCURACY
      - RELEVANCE
      - COMPLETENESS
      - USEFULNESS
    Returns metric scores + overall + detailed feedback.
    """

    final_text = extract_final_answer_text(response_obj)

    if not isinstance(final_text, str) or not final_text.strip():
        return {
            "passed": False,
            "reason": "Final answer text could not be extracted from response messages.",
            "metrics": {"accuracy": 0.0, "relevance": 0.0, "completeness": 0.0, "usefulness": 0.0, "overall": 0.0},
            "feedback": {"missing": ["no_final_answer"], "notes": []}
        }

    # --- RELEVANCE (token overlap between question and answer) ---
    q_tokens = set(tokenize(question))
    a_tokens = set(tokenize(final_text))
    overlap = len(q_tokens & a_tokens) / max(1, len(q_tokens))
    relevance = clamp(100.0 * overlap)

    # --- COMPLETENESS (expected aspects coverage) ---
    expected_aspects = parse_expected_aspects(expected_response)
    detected = detect_aspects_in_response(final_text)

    missing_aspects = []
    covered = 0
    total_expected = 0
    for aspect, is_expected in expected_aspects.items():
        if is_expected:
            total_expected += 1
            if detected.get(aspect, False):
                covered += 1
            else:
                missing_aspects.append(aspect)

    # Base completeness: % of expected aspects covered
    completeness = clamp(100.0 * (covered / max(1, total_expected))) if total_expected else 70.0
    # Bonus if response has structure (bullets/sections)
    structured = bool(re.search(r"(\n- |\n\d+\.|###|\bhere's why\b|\breasons?\b)", final_text.lower()))
    if structured:
        completeness = clamp(completeness + 10)

    # --- USEFULNESS (actionable + quantified) ---
    actionable = detected["actions"]
    quantified = detected["quantified"]
    # Heuristic score
    usefulness = 40.0
    if actionable:
        usefulness += 30.0
    if quantified:
        usefulness += 30.0
    usefulness = clamp(usefulness)

    # --- ACCURACY (heuristic: groundedness & internal consistency) ---
    # We don’t have a gold answer, but we can penalize obvious non-answers and reward grounded numeric/tool-aware content.
    lower = final_text.lower()
    accuracy = 55.0

    # Penalize refusals/errors
    if "error" in lower or "cannot" in lower or "can't" in lower:
        accuracy = 15.0

    # Reward mention of pricing concepts when question is cost-related
    if any(k in lower for k in ["off-peak", "peak", "rate", "per kwh", "pricing", "$"]):
        accuracy += 15.0

    # Reward quantified statements (numbers, $)
    if quantified:
        accuracy += 20.0

    # Penalize if answer has no actionable content at all
    if not actionable:
        accuracy -= 15.0

    accuracy = clamp(accuracy)

    # --- OVERALL (weighted) ---
    overall = clamp(0.30 * accuracy + 0.30 * relevance + 0.20 * completeness + 0.20 * usefulness)
    passed = overall >= 60

    feedback_notes = []
    if missing_aspects:
        feedback_notes.append(f"Missing expected aspects: {missing_aspects}")
    if not structured:
        feedback_notes.append("Consider using a clearer structure (bullets/steps) for completeness.")
    if not quantified and expected_aspects.get("cost", False):
        feedback_notes.append("Consider adding a simple quantified savings example for usefulness.")

    return {
        "passed": passed,
        "reason": "Response metrics computed.",
        "metrics": {
            "accuracy": round(accuracy, 1),
            "relevance": round(relevance, 1),
            "completeness": round(completeness, 1),
            "usefulness": round(usefulness, 1),
            "overall": round(overall, 1)
        },
        "final_answer": final_text,
        "feedback": {
            "expected_aspects": expected_aspects,
            "detected_aspects": {k: detected[k] for k in ["time","cost","solar","actions","quantified"]},
            "missing_aspects": missing_aspects,
            "notes": feedback_notes
        }
    }


In [14]:
# TODO: Create a tool udage evaluator


def evaluate_tool_usage(messages_list: List[Any], expected_tools: List[str]) -> Dict[str, Any]:
    """
    Required tool metrics (0–100):
      - Tool Appropriateness: Were the right tools selected for the task?
      - Tool Completeness: Were all necessary tools used?
    Also returns comprehensive feedback: used/matched/missing/extra + narrative.
    """

    expected_set = set(expected_tools or [])
    used_tools = extract_used_tools_from_messages(messages_list)
    used_set = set(used_tools)

    if not expected_set:
        return {
            "passed": True,
            "reason": "No expected tools specified; tool evaluation skipped.",
            "metrics": {"tool_appropriateness": 100.0, "tool_completeness": 100.0, "overall": 100.0},
            "details": {"used_tools": sorted(list(used_set)), "matched_tools": [], "missing_tools": [], "extra_tools": []},
            "feedback": ["No expected tools provided, so tool usage is not evaluated."]
        }

    matched = sorted(list(expected_set & used_set))
    missing = sorted(list(expected_set - used_set))
    extra = sorted(list(used_set - expected_set))

    # Tool Completeness: % expected tools used
    tool_completeness = clamp(100.0 * len(matched) / max(1, len(expected_set)))

    # Tool Appropriateness: precision-like measure = matched / (used) with mild tolerance for 1 extra tool
    if len(used_set) == 0:
        tool_appropriateness = 0.0
    else:
        precision = len(matched) / max(1, len(used_set))
        tool_appropriateness = 100.0 * precision

        # mild penalty: allow 1 extra tool without heavy hit
        if len(extra) == 1:
            tool_appropriateness = clamp(tool_appropriateness + 10)  # soften penalty
        if len(extra) > 1:
            tool_appropriateness = clamp(tool_appropriateness - 10 * (len(extra) - 1))

    tool_appropriateness = clamp(tool_appropriateness)

    overall = clamp(0.55 * tool_completeness + 0.45 * tool_appropriateness)
    passed = overall >= 60

    feedback = []
    if not used_set:
        feedback.append("No tools detected in execution trace. The agent likely answered without tools or tools were not wired correctly.")
    else:
        feedback.append(f"Tools used: {sorted(list(used_set))}")
        feedback.append(f"Expected tools: {sorted(list(expected_set))}")

    if missing:
        feedback.append(f"Missing expected tools (reduce completeness): {missing}")
    else:
        feedback.append("All expected tools were used (completeness OK).")

    if extra:
        feedback.append(f"Extra tools used (may reduce appropriateness): {extra}")
    else:
        feedback.append("No extra tools used (appropriateness OK).")

    # Suggest improvements when failing
    if not passed:
        if missing:
            feedback.append("Recommendation: Update agent prompt/logic to ensure missing tools are invoked for this scenario.")
        if extra:
            feedback.append("Recommendation: Avoid unnecessary tools unless they add clear value to the final answer.")

    return {
        "passed": passed,
        "reason": "Tool usage metrics computed.",
        "metrics": {
            "tool_appropriateness": round(tool_appropriateness, 1),
            "tool_completeness": round(tool_completeness, 1),
            "overall": round(overall, 1)
        },
        "details": {
            "used_tools": sorted(list(used_set)),
            "matched_tools": matched,
            "missing_tools": missing,
            "extra_tools": extra
        },
        "feedback": feedback
    }



In [15]:

def display_evaluation_report(report: Dict[str, Any]) -> None:
    print("\n" + "=" * 90)
    print("ECOHOME ENERGY ADVISOR — COMPREHENSIVE EVALUATION REPORT")
    print("=" * 90)
    print(f"Overall Score (All test cases combined): {report['overall_score']}/100")
    print(f"Total test cases: {report['total_test_cases']}\n")

    print("Average Response Metrics (0–100):")
    for k, v in report["avg_response_metrics"].items():
        print(f"  - {k.upper():<12}: {v}")

    print("\nAverage Tool Metrics (0–100):")
    for k, v in report["avg_tool_metrics"].items():
        print(f"  - {k.upper():<20}: {v}")

    print("\nStrengths:")
    for s in report["strengths"]:
        print(f"  - {s}")

    print("\nWeaknesses:")
    for w in report["weaknesses"]:
        print(f"  - {w}")

    print("\nRecommendations:")
    for r in report["recommendations"]:
        print(f"  - {r}")

    if report.get("lowest_scoring_tests"):
        print("\nLowest Scoring Test Cases:")
        for row in report["lowest_scoring_tests"]:
            print(f"  - {row['test_id']}: overall={row['overall']}, response={row['response_overall']}, tools={row['tool_overall']}")

    print("=" * 90 + "\n")


In [16]:
# TODO: Generate a comprehensive evaluation report
# Calculate overall scores and metrics
# Identify strengths and weaknesses
# Provide recommendations for improvement


def generate_evaluation_report(evaluation_results: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Aggregates metrics across all tests into a single structured report.
    Produces ONE overall score and explicit strengths/weaknesses/recommendations. [2](https://ts.accenture.com/sites/ambgwiki/Microsoft%20NDA%20Documents/Azure%20OpenAI%20Service%20L100%20Pitch%20Deck.PDF?web=1)[1](https://github.com/MariusJochheim/Udacity-Master-Energy-Advisor/blob/main/03_run_and_evaluate.ipynb)
    """

    if not evaluation_results:
        raise ValueError("No evaluation results provided.")

    def avg(vals): return round(sum(vals) / max(1, len(vals)), 1)

    # Aggregate response metrics
    acc, rel, comp, use, resp_over = [], [], [], [], []
    tool_app, tool_comp, tool_over = [], [], []
    per_test = []

    for r in evaluation_results:
        tid = r.get("test_id", "unknown")
        rm = (r.get("response_eval") or {}).get("metrics", {})
        tm = (r.get("tool_eval") or {}).get("metrics", {})

        acc.append(rm.get("accuracy", 0.0))
        rel.append(rm.get("relevance", 0.0))
        comp.append(rm.get("completeness", 0.0))
        use.append(rm.get("usefulness", 0.0))
        resp_over.append(rm.get("overall", 0.0))

        tool_app.append(tm.get("tool_appropriateness", 0.0))
        tool_comp.append(tm.get("tool_completeness", 0.0))
        tool_over.append(tm.get("overall", 0.0))

        combined = round(0.7 * rm.get("overall", 0.0) + 0.3 * tm.get("overall", 0.0), 1)
        per_test.append({
            "test_id": tid,
            "overall": combined,
            "response_overall": rm.get("overall", 0.0),
            "tool_overall": tm.get("overall", 0.0)
        })

    avg_response_metrics = {
        "accuracy": avg(acc),
        "relevance": avg(rel),
        "completeness": avg(comp),
        "usefulness": avg(use),
        "overall": avg(resp_over)
    }

    avg_tool_metrics = {
        "tool_appropriateness": avg(tool_app),
        "tool_completeness": avg(tool_comp),
        "overall": avg(tool_over)
    }

    # ✅ ONE overall score across all test cases combined
    overall_score = round(0.7 * avg_response_metrics["overall"] + 0.3 * avg_tool_metrics["overall"], 1)

    # Strengths/weaknesses based on thresholds
    strengths, weaknesses, recommendations = [], [], []

    def add_strength_if(metric_name, val, threshold=75):
        if val >= threshold:
            strengths.append(f"{metric_name} is strong (avg {val}/100).")

    def add_weakness_if(metric_name, val, threshold=75, rec=None):
        if val < threshold:
            weaknesses.append(f"{metric_name} needs improvement (avg {val}/100).")
            if rec:
                recommendations.append(rec)

    add_strength_if("ACCURACY", avg_response_metrics["accuracy"])
    add_strength_if("RELEVANCE", avg_response_metrics["relevance"])
    add_strength_if("COMPLETENESS", avg_response_metrics["completeness"])
    add_strength_if("USEFULNESS", avg_response_metrics["usefulness"])
    add_strength_if("Tool Appropriateness", avg_tool_metrics["tool_appropriateness"])
    add_strength_if("Tool Completeness", avg_tool_metrics["tool_completeness"])

    add_weakness_if("ACCURACY", avg_response_metrics["accuracy"],
                    rec="Improve ACCURACY by citing tool outputs (rates, hours, kWh) and showing short calculations.")
    add_weakness_if("RELEVANCE", avg_response_metrics["relevance"],
                    rec="Improve RELEVANCE by answering in the first 1–2 lines before adding reasoning.")
    add_weakness_if("COMPLETENESS", avg_response_metrics["completeness"],
                    rec="Improve COMPLETENESS by covering all expected aspects (time + cost + solar/etc.) and using bullet steps.")
    add_weakness_if("USEFULNESS", avg_response_metrics["usefulness"],
                    rec="Improve USEFULNESS by including concrete time windows/settings and a small quantified savings example.")
    add_weakness_if("Tool Appropriateness", avg_tool_metrics["tool_appropriateness"],
                    rec="Improve tool appropriateness by avoiding unnecessary tools unless they add value.")
    add_weakness_if("Tool Completeness", avg_tool_metrics["tool_completeness"],
                    rec="Improve tool completeness by ensuring expected tools are always invoked for each scenario type.")

    if not strengths:
        strengths.append("Evaluation successfully computed all required metrics across all test cases.")
    if not weaknesses:
        weaknesses.append("No major weaknesses detected based on current thresholds.")
    if not recommendations:
        recommendations.append("Continue adding edge-case test cases and refine prompt to improve consistency.")

    # Lowest scoring tests (top 3)
    lowest_scoring_tests = sorted(per_test, key=lambda x: x["overall"])[:3]

    return {
        "overall_score": overall_score,
        "total_test_cases": len(evaluation_results),
        "avg_response_metrics": avg_response_metrics,
        "avg_tool_metrics": avg_tool_metrics,
        "strengths": strengths,
        "weaknesses": weaknesses,
        "recommendations": recommendations,
        "lowest_scoring_tests": lowest_scoring_tests
    }



In [17]:


evaluation_results = []

for r in test_results:
    response_obj = r["response"]
    messages_list = response_obj.get("messages", []) if isinstance(response_obj, dict) else []

    resp_eval = evaluate_response(
        question=r["question"],
        response_obj=response_obj,
        expected_response=r["expected_response"]
    )

    tool_eval = evaluate_tool_usage(
        messages_list=messages_list,
        expected_tools=r["expected_tools"]
    )

    evaluation_results.append({
        "test_id": r["test_id"],
        "response_eval": resp_eval,
        "tool_eval": tool_eval
    })

report = generate_evaluation_report(evaluation_results)
display_evaluation_report(report)




ECOHOME ENERGY ADVISOR — COMPREHENSIVE EVALUATION REPORT
Overall Score (All test cases combined): 85.4/100
Total test cases: 10

Average Response Metrics (0–100):
  - ACCURACY    : 81.0
  - RELEVANCE   : 81.8
  - COMPLETENESS: 100.0
  - USEFULNESS  : 91.0
  - OVERALL     : 87.0

Average Tool Metrics (0–100):
  - TOOL_APPROPRIATENESS: 77.7
  - TOOL_COMPLETENESS   : 85.0
  - OVERALL             : 81.7

Strengths:
  - ACCURACY is strong (avg 81.0/100).
  - RELEVANCE is strong (avg 81.8/100).
  - COMPLETENESS is strong (avg 100.0/100).
  - USEFULNESS is strong (avg 91.0/100).
  - Tool Appropriateness is strong (avg 77.7/100).
  - Tool Completeness is strong (avg 85.0/100).

Weaknesses:
  - No major weaknesses detected based on current thresholds.

Recommendations:
  - Continue adding edge-case test cases and refine prompt to improve consistency.

Lowest Scoring Test Cases:
  - cost_savings_1: overall=73.2, response=81.2, tools=54.5
  - solar_2: overall=77.5, response=67.8, tools=100.0
  -